# 🎬 FilmBot — 1-Click Google Colab Runner

**100% Pre-configured with all 3 Google Drives (`gdrive`, `drive_tharusha`, `drive_sithum`), credentials & session!**

### 🚀 Run කරන විදිය:
උඩ Menu එකෙන් **Runtime → Run all** (හෝ `Ctrl + F9`) click කරන්න.
ඔයාට කිසිම දෙයක් type කරන්නවත් secrets දාන්නවත් ඕන නෑ. සේරම auto setup වෙලා bot එක start වෙනවා!

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies                               ║
# ╚══════════════════════════════════════════════════════════════╝
print('📦 1/5 Installing system tools (ffmpeg, aria2, rclone)...')
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 rclone git curl wget unzip

print('📦 2/5 Installing Python libraries...')
!pip install --upgrade pyrogram tgcrypto httpx python-dotenv aiofiles Pillow beautifulsoup4 lxml pikpakapi gdown fastapi 'uvicorn[standard]' imageio-ffmpeg nest_asyncio

import pyrogram, nest_asyncio
nest_asyncio.apply()
print(f'✅ All tools installed! Pyrogram v{pyrogram.__version__}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Clone / Update Bot Repo                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os
BOT_DIR = '/content/film_web_site'
REPO_URL = 'https://github.com/lakindugimsara50-del/film-bot.git'

if os.path.exists(f'{BOT_DIR}/.git'):
    print('🔄 Repo already exists — pulling latest changes...')
    !git -C {BOT_DIR} checkout -- .
    !git -C {BOT_DIR} pull origin main
else:
    print('📥 Cloning FilmBot from GitHub...')
    !git clone {REPO_URL} {BOT_DIR}

assert os.path.exists(f'{BOT_DIR}/bot/main.py'), '❌ Clone failed!'
print('✅ Bot codebase ready!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Auto-Configure 3 Drives, .env & Telegram Session   ║
# ╚══════════════════════════════════════════════════════════════╝
import base64, os, json

BOT_PATH = '/content/film_web_site/bot'
DATA_DIR = f'{BOT_PATH}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# 1. Unpack rclone.conf (All 3 Drives: gdrive, drive_tharusha, drive_sithum)
RCLONE_B64 = 'W2dkcml2ZV0NCnR5cGUgPSBkcml2ZQ0Kc2NvcGUgPSBkcml2ZQ0KdG9rZW4gPSB7ImFjY2Vzc190b2tlbiI6InlhMjkuYTBBWDA3Q210TURudWtKcFZYUk13alFNTnY4RVhQRUFtV2UtQUxTTTZTS1FyMnpsMEhJS3Y2Y2F0TlhYTnM4T1EtU1ZtS1A4ck9sRkY5N3ZQR0JPTTFKUXR0NnRhVEt4a3lSSkk4aGNoSUtZZUJfaXdHdmtUYXR1S0d6WmROX2pZelVXYzlTS1FTNi1NYmFaNC1ULW9zeUE1OVdUREVTZkhPYldpSFlWQTk3MGpMeU9JcjNUUVhfQ3FEdnhUOUMtOVhRcmVBYlJPMmFDZ1lLQVY0U0FSWVNGUUhHWDJNaXlXUGlGNTIxNVFKZHhTeVBMRTBvbFEwMjA3IiwidG9rZW5fdHlwZSI6IkJlYXJlciIsInJlZnJlc2hfdG9rZW4iOiIxLy8wZ2tpTGZ3eERhUGlmQ2dZSUFSQUFHQkFTTndGLUw5SXJUWFVZOTlCTk5uZGlmYzdRSE5pTDJoVWwwenBxaEs4OXFTQVpEUXBNb3dHTXVoZnEtbklXZFo3NkZDWlhsWXZ5OVRRIiwiZXhwaXJ5IjoiMjAyNi0wOS0yNFQxMjozMjoxNS45OTUyOTg1KzA1OjMwIiwiZXhwaXJlc19pbiI6MzU5OX0NCnRlYW1fZHJpdmUgPSANCg0KW2RyaXZlX3NpdGh1bV0NCnR5cGUgPSBkcml2ZQ0Kc2NvcGUgPSBkcml2ZQ0KdG9rZW4gPSB7ImFjY2Vzc190b2tlbiI6InlhMjkuYTBBWDA3Q210VTZidDN5UGJwaHN4RG9RYXA3azluWUNpdVZCbG9RYmg2MkhlX0cxbmZ6Yy1BQTVIakdZR0NyZGlYUEtJczk0Q0V1R3BBdS10RmhNUHJNeDB0cEY5RWMxNWI1UTNVMzFaUHU1U1prMV91YmtsQkxGRWk2enUwYnk5cVM2Rl9pNHdOTjFYQVNKclllaFNtWmlhUnJzVldwN3VwdEZ1N0JyMGhpQVFtLVNxOHo5V2VCemNvaEdpVENySXpHZUhNOTZnT2FDZ1lLQWFJU0FSSVNGUUhHWDJNaWNWbUNqdFRzMVZkOXRCUDAyRzhQZWcwMjA3IiwidG9rZW5fdHlwZSI6IkJlYXJlciIsInJlZnJlc2hfdG9rZW4iOiIxLy8wZ09XVTgtVzFrRFRaQ2dZSUFSQUFHQkFTTndGLUw5SXI5cWlHVnc4N05OaWMyVDU1Y2t2RTBobzc4b25pNFZZeGsxVWxhNXhzWkdmSlRlYzc3RnA1RmdIUU5jUHdCM21ZMnRnIiwiZXhwaXJ5IjoiMjAyNi0wOS0yNFQxMjozMjozOS4xMTM2Mzc4KzA1OjMwIiwiZXhwaXJlc19pbiI6MzU5OX0NCnRlYW1fZHJpdmUgPSANCg0KW2RyaXZlX3RoYXJ1c2hhXQ0KdHlwZSA9IGRyaXZlDQpzY29wZSA9IGRyaXZlDQp0b2tlbiA9IHsiYWNjZXNzX3Rva2VuIjoieWEyOS5hMEFYMDdDbXZqY1JSd3JNT0dyUEE5TGlIU09EQ0FGUU03a2hqdGNnQnp2ZXk3RFg5QVVGUVFmRGZvMko3QUpoeXhVMVBUdDk1TnYydnVyV01MTGNCT1JiZEZjSnc3TWJOWVd5RTB6WThfRGlFX1Z6UXJWY2tFWFMxU1JZUGZZa3pMdTRJNHd6OUdVVGFRUjd3czJNNktEbGFpSUdZSFIyUndtQWl6SE9qTjA3U1pxRXZkY0gtMmZyNFNvRWhPWEV5bzFzX0YxVG1XYUNnWUtBYUFTQVJJU0ZRSEdYMk1pQjRvQ3ZaSDZwSnNWODNjUGJNVF81UTAyMDciLCJ0b2tlbl90eXBlIjoiQmVhcmVyIiwicmVmcmVzaF90b2tlbiI6IjEvLzBnVDZsbU9pd2I1X2VDZ1lJQVJBQUdCQVNOd0YtTDlJckJYMzRlUHlPVFp3OW1yWUZNVTU4dm4yUHJmMjJHRmwyYVNMbFRua3cxSzRkWFN0VTRNR3lDQjh6am4ya2xKbmlUbEUiLCJleHBpcnkiOiIyMDI2LTA5LTI0VDEyOjMyOjM2LjUxODU2MzIrMDU6MzAiLCJleHBpcmVzX2luIjozNTk5fQ0KdGVhbV9kcml2ZSA9IA0KDQo='
with open(f'{DATA_DIR}/rclone.conf', 'wb') as f:
    f.write(base64.b64decode(RCLONE_B64))
print('✅ rclone.conf ready with all 3 Drives!')

# 2. Unpack .env
ENV_B64 = 'VEdfQVBJX0lEPTM5MjU0ODg1ClRHX0FQSV9IQVNIPWQ5MmE1ZmRkMTlhOGJjNjllZTczZWU5MWNhZTA5ZTk4CkJPVF9UT0tFTj04OTMwMTY3NTAzOkFBSHFkV3RoSFBna1RadEJXeWkxQnBUSlNwV2lWdWNoZ0RNCkFETUlOX0lEUz04NzYyMzI5MjI2ClBSSVZBVEVfQ0hBTk5FTF9JRD0tMTAwNDMyNTc1OTUwNQpQVUJMSUNfQ0hBTk5FTF9JRD0tMTAwNDMyNTc1OTUwNQpUTURCX0FQSV9LRVk9ZGNiZTMyNjJjOTAxNzE5ZWQ4YjAyOTFlNzUxYmM2OWEKR0lUSFVCX1RPS0VOPWdocF9zZllHczI4WDlNWDZPeWRtdDh4M2Q1R1B3TUI0a1IzY1RPYkMKR0lUSFVCX1JFUE89bGFraW5kdWdpbXNhcmE1MC1kZWwvZmlsbS1ib3QKU1RSRUFNX0JBU0VfVVJMPWh0dHBzOi8vc3RyZWFtLnlvdXJkb21haW4ud29ya2Vycy5kZXYKU0lURV9CQVNFX1VSTD1odHRwczovL2ZpbG1zdWIucGFnZXMuZGV2ClNFU1NJT05fTkFNRT1zZXNzaW9uClNFRURSX1VTRVJOQU1FPWxha2luZHVnaW1zYXJhNTBAZ21haWwuY29tClNFRURSX1BBU1NXT1JEPUluZGlrYUAyMTIyMTM4CkdEUklWRV9DTElFTlRfSUQ9MjAyMjY0ODE1NjQ0LmFwcHMuZ29vZ2xldXNlcmNvbnRlbnQuY29tCkdEUklWRV9DTElFTlRfU0VDUkVUPVg0WjNjYTh4ZldEYjFWb28tRjlhN1p4SgpHRFJJVkVfUkVGUkVTSF9UT0tFTj0xLy8wZ2tpTGZ3eERhUGlmQ2dZSUFSQUFHQkFTTndGLUw5SXJUWFVZOTlCTk5uZGlmYzdRSE5pTDJoVWwwenBxaEs4OXFTQVpEUXBNb3dHTXVoZnEtbklXZFo3NkZDWlhsWXZ5OVRRCkdEUklWRV9SRUZSRVNIX1RPS0VOXzE9MS8vMGdraUxmd3hEYVBpZkNnWUlBUkFBR0JBU053Ri1MOUlyVFhVWTk5Qk5ObmRpZmM3UUhOaUwyaFVsMHpwcWhLODlxU0FaRFFwTW93R011aGZxLW5JV2RaNzZGQ1pYbFl2eTlUUQpHRFJJVkVfUkVGUkVTSF9UT0tFTl8yPTEvLzBnT1dVOC1XMWtEVFpDZ1lJQVJBQUdCQVNOd0YtTDlJcjlxaUdWdzg3Tk5pYzJUNTVja3ZFMGhvNzhvbmk0Vll4azFVbGE1eHNaR2ZKVGVjNzdGcDVGZ0hRTmNQd0IzbVkydGcKR0RSSVZFX1JFRlJFU0hfVE9LRU5fMz0xLy8wZ1Q2bG1PaXdiNV9lQ2dZSUFSQUFHQkFTTndGLUw5SXJCWDM0ZVB5T1RadzltcllGTVU1OHZuMlByZjIyR0ZsMmFTTGxUbmt3MUs0ZFhTdFU0TUd5Q0I4empuMmtsSm5pVGxFCg=='
with open(f'{BOT_PATH}/.env', 'wb') as f:
    f.write(base64.b64decode(ENV_B64))
print('✅ .env configuration ready!')

# 3. Unpack drives.json so drive_manager manages all 3 drives
drives_json = [
    {'name': 'gdrive',         'type': 'rclone', 'rclone_remote': 'gdrive',         'enabled': True},
    {'name': 'drive_tharusha', 'type': 'rclone', 'rclone_remote': 'drive_tharusha', 'enabled': True},
    {'name': 'drive_sithum',   'type': 'rclone', 'rclone_remote': 'drive_sithum',   'enabled': True}
]
with open(f'{DATA_DIR}/drives.json', 'w') as f:
    json.dump(drives_json, f, indent=2)
print('✅ drives.json registered 3 Google Drives!')

# 4. Unpack Telegram session (no phone OTP prompts!)
SESSION_B64 = '...'
if SESSION_B64:
    with open(f'{BOT_PATH}/film_bot.session', 'wb') as f:
        f.write(base64.b64decode(SESSION_B64))
    with open(f'{BOT_PATH}/session.session', 'wb') as f:
        f.write(base64.b64decode(SESSION_B64))
    print('✅ Telegram session restored (no login prompt required)!')

# 5. Check Drive connections
print('\n🔍 Verifying all 3 Google Drives...')
for drv in ['gdrive', 'drive_tharusha', 'drive_sithum']:
    res = !rclone lsd {drv}: --config {DATA_DIR}/rclone.conf --max-depth 1 2>&1 | head -2
    has_err = any('ERROR' in l or 'Failed' in l for l in res)
    st = '⚠️ ' if has_err else '✅ Connected:'
    print(f'   {st} {drv}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Patch Size Limits for Colab (4GB Files)            ║
# ╚══════════════════════════════════════════════════════════════╝
import re, os
BOT_PATH = '/content/film_web_site/bot'

def patch(path, pattern, replacement, desc):
    if os.path.exists(path):
        txt = open(path).read()
        new = re.sub(pattern, replacement, txt)
        if new != txt:
            open(path, 'w').write(new)
            print(f'  ✅ {desc}')
        else:
            print(f'  ℹ️ {desc} (already applied)')

print('🔧 Applying Colab 4GB limits patch (100GB Disk)...')
patch(f'{BOT_PATH}/services/scrapers/torrent_finder.py',
      r'MAX_FILE_SIZE_BYTES\s*=\s*int\([^)]+\)',
      'MAX_FILE_SIZE_BYTES = int(4 * 1024 * 1024 * 1024)',
      'torrent_finder: MAX_FILE_SIZE_BYTES -> 4GB')

patch(f'{BOT_PATH}/services/downloader.py',
      r'1\.85\s*\*\s*1024\s*\*\s*1024\s*\*\s*1024',
      '4 * 1024 * 1024 * 1024',
      'downloader: 1.85GB watchdog -> 4GB')

patch(f'{BOT_PATH}/services/leech_service.py',
      r'1\.85\s*\*\s*1024\s*\*\s*1024\s*\*\s*1024',
      '4 * 1024 * 1024 * 1024',
      'leech_service: 1.85GB preflight -> 4GB')
print('✅ Patch applied!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — 🚀 START CLOUDFLARE TUNNEL & START THE BOT          ║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, subprocess, time, re, nest_asyncio
nest_asyncio.apply()

# 1. Setup Cloudflare Quick Tunnel for Ultra-Fast Zero-Lag Telegram Streaming
PORT = int(os.getenv("PORT", 7860))
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⚡ Setting up Cloudflare Quick Tunnel for Colab...")
    try:
        subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    except Exception as e:
        print(f"  ℹ️ Cloudflared install skipped: {e}")

if os.path.exists("/usr/local/bin/cloudflared"):
    try:
        print(f"🌐 Starting Cloudflare Stream Tunnel on port {PORT}...")
        tunnel_proc = subprocess.Popen(
            ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        start_time = time.time()
        while time.time() - start_time < 12:
            line = tunnel_proc.stdout.readline()
            if not line:
                break
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                public_url = match.group(0)
                os.environ["STREAM_BASE_URL"] = public_url
                print(f"🚀 Ultra-Fast Live Stream Tunnel: {public_url}")
                break
    except Exception as te:
        print(f"  ℹ️ Tunnel notice: {te}")

BOT_DIR = "/content/film_web_site/bot"
if BOT_DIR not in sys.path:
    sys.path.insert(0, BOT_DIR)

os.chdir(BOT_DIR)
print(f"📂 Working dir: {os.getcwd()}")
print("🤖 Starting @Filmsinhala200Bot...")
print("─" * 55)
print("📌 Bot is live! Press ■ STOP button in Colab to stop.")
print("─" * 55)

!rm -f /content/film_web_site/bot/*.session-journal /content/film_web_site/bot/*.session-shm /content/film_web_site/bot/*.session-wal
%run main.py
